# CSPF2 Dummy-Covariate Analysis (`Cat1 == toys games`)

This notebook:
1. Runs `examples/test_cspf_new2.py`.
2. Re-fits the same model in-notebook.
3. Reports covariate-effect estimates and shrinkage variance components.
4. Provides a scientific interpretation focused on effect sizes and uncertainty.

In [ ]:
import subprocess
import sys

cmd = ["poetry", "run", "python", "examples/test_cspf_new2.py"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Script failed with exit code {result.returncode}")

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import jax.numpy as jnp
from sklearn.feature_extraction.text import CountVectorizer
from IPython.display import Markdown, display

from poisson_topicmodels.models.CSPF2 import CSPF2
from examples.test_cspf_new2 import _build_keywords

## Refit for Parameter Inspection

In [ ]:
np.random.seed(42)

df = pd.read_csv("./data/10k_amazon.csv")
df = df.dropna(subset=["Text", "Cat1"]).copy()
n_docs = min(2000, len(df))
df = df.sample(n=n_docs, random_state=42).reset_index(drop=True)

vectorizer = CountVectorizer(stop_words="english", min_df=5, max_features=5000)
counts = sparse.csr_matrix(vectorizer.fit_transform(df["Text"]), dtype=np.float32)
vocab = vectorizer.get_feature_names_out()
keywords = _build_keywords(set(vocab))

x_design = pd.DataFrame(
    {
        "cat1::is_toys_games": (df["Cat1"].astype(str).str.lower() == "toys games").astype(np.float32)
    }
)

model = CSPF2(
    counts=counts,
    vocab=vocab,
    keywords=keywords,
    residual_topics=2,
    batch_size=min(256, counts.shape[0]),
    X_design_matrix=x_design,
)

_ = model.train_step(num_steps=30, lr=0.01, random_seed=7)
params = model.estimated_params

print("Final loss:", model.Metrics.loss[-1])
print("Finite loss:", np.isfinite(model.Metrics.loss[-1]))

## Covariate Effect and Variance Output

In [ ]:
topic_names = list(model.keywords.keys()) + [f"residual_topic_{i+1}" for i in range(model.residual_topics)]
x = x_design.iloc[:, 0].values
n1 = float(x.sum())
p1 = float(x.mean())
scaling = float(model.group_scaling_diag[0])

lambda_mean = np.asarray(params["lambda_location"])[0]
lambda_sd = np.asarray(params["lambda_scale"])[0]
lambda_l95 = lambda_mean - 1.96 * lambda_sd
lambda_u95 = lambda_mean + 1.96 * lambda_sd
significant = (lambda_l95 > 0) | (lambda_u95 < 0)

int_mean = np.asarray(params["lambda_intercept_location"])
mu_x0 = np.log1p(np.exp(int_mean))
mu_x1 = np.log1p(np.exp(int_mean + lambda_mean))
mu_ratio = mu_x1 / np.maximum(mu_x0, 1e-12)

tau2_shape = np.asarray(params["tau2_shape"])
tau2_rate = np.asarray(params["tau2_rate"])
tau2_mean = tau2_shape / tau2_rate
tau2_var = tau2_shape / (tau2_rate ** 2)

delta2_shape = np.asarray(params["delta2_shape"])[0]
delta2_rate = np.asarray(params["delta2_rate"])[0]
delta2_mean = delta2_shape / delta2_rate
delta2_var = delta2_shape / (delta2_rate ** 2)

implied_var_lambda = tau2_mean * delta2_mean * scaling

summary_df = pd.DataFrame(
    {
        "topic": topic_names,
        "lambda_mean": lambda_mean,
        "lambda_sd": lambda_sd,
        "lambda_l95": lambda_l95,
        "lambda_u95": lambda_u95,
        "ci_excludes_zero": significant,
        "mu_x0": mu_x0,
        "mu_x1": mu_x1,
        "mu_ratio_x1_vs_x0": mu_ratio,
        "tau2_mean": tau2_mean,
        "tau2_var": tau2_var,
        "delta2_mean": delta2_mean,
        "delta2_var": delta2_var,
        "implied_prior_var_lambda": implied_var_lambda,
    }
).sort_values("lambda_mean", ascending=False)

print(f"N documents: {len(df)}")
print(f"Dummy prevalence P(Cat1='toys games'): {p1:.4f} ({int(n1)} docs)")
print(f"Design scaling (X_g'X_g)^(-1) diagonal for dummy: {scaling:.6f}")
display(summary_df.round(4))

top_pos = summary_df.iloc[0]
top_neg = summary_df.iloc[-1]
n_sig = int(summary_df["ci_excludes_zero"].sum())

interpretation = f"""
### Scientific Interpretation

- **Design prevalence and scaling:** The dummy covariate is active in **{p1:.2%}** of documents ({int(n1)} / {len(df)}). The design-adaptive scaling term is **{scaling:.6f}**, which is approximately `1/n1` for this one-hot column.
- **Covariate effect (`lambda`) on topic prevalence:** The strongest positive estimated effect is for **{top_pos['topic']}** (`lambda_mean={top_pos['lambda_mean']:.3f}`, 95% CI [{top_pos['lambda_l95']:.3f}, {top_pos['lambda_u95']:.3f}]). The strongest negative effect is for **{top_neg['topic']}** (`lambda_mean={top_neg['lambda_mean']:.3f}`, 95% CI [{top_neg['lambda_l95']:.3f}, {top_neg['lambda_u95']:.3f}]).
- **Uncertainty/significance pattern:** **{n_sig} of {len(summary_df)}** topic-specific effects have 95% intervals excluding 0. This indicates how many topic shifts are credibly associated with `Cat1 = toys games` under the variational approximation.
- **Effect on expected topic intensity mean:** The multiplicative shift in the Gamma mean parameter, `mu(x=1)/mu(x=0)`, quantifies practical effect size. Values above 1 indicate higher expected topic intensity in `toys games` reviews; values below 1 indicate lower intensity.
- **Variance components and shrinkage:** `tau2` (global, topic-level) and `delta2` (local, group-topic level) jointly control effect dispersion. Smaller implied prior variance (`tau2_mean * delta2_mean * scaling`) indicates stronger shrinkage of the covariate effect toward zero; larger values indicate weaker shrinkage and more topic-specific flexibility.
"""
display(Markdown(interpretation))